# LLM Repair Evaluation

Evaluate LLM-based repair and translation quality.

In [48]:
sys.path.append(os.path.join(os.getcwd(), "..", "src"))
import os, json, sys, datetime
from openai import OpenAI
from repair import PROMPTS
import sacrebleu
import pandas as pd


## Preparing the source data

In [3]:
# Importing gold set to list of dictionaries
with open("gold_set.jsonl", "r", encoding="utf-8") as file:
    gold_set = [json.loads(row) for row in file if row.strip()]
    
print(gold_set[1])

{'case_id': 'f02', 'category': 'fuzzy_real', 'query': 'plague ($type)', 'reference': 'zaraza ($type)', 'edit_type': 'addition', 'requires_agreement': 'no', 'flag': 'ok', 'comment': None, 'previous_source': 'plague', 'target_approved': 'zaraza', 'source_file': 'pl_help.po'}


In [5]:
necessary_fields = [
    "case_id",
    "category",
    "edit_type",
    "query",
    "reference"
]

additional_fields = [
    "target",
    "target_approved"
]

gold_set_filtered = []

for record in gold_set:
    new_record = {
        field: record.get(field)
        for field in necessary_fields
    }

    if record.get("category") == "edited" or record.get("category") == "fuzzy_real":
        new_record.update({
            field: record.get(field)
            for field in additional_fields
        })

    gold_set_filtered.append(new_record)
    
print(gold_set_filtered[0])

{'case_id': 'f01', 'category': 'fuzzy_real', 'edit_type': 'reworded', 'query': 'Pirate Flagship', 'reference': 'Okręt flagowy piratów', 'target': None, 'target_approved': 'Ogniste widmo'}


## Generating target translation

In [7]:
# Checking if a given version was already translated to avoid repeated generation
with open("prompt_runs.jsonl", "r", encoding="utf-8") as file:
    prompt_runs = [json.loads(row) for row in file if row.strip()]
    
print(prompt_runs[1])

{'ts': '2026-08-09T16:50:03', 'prompt_version': 'v1_only_different', 'model': 'gpt-4o-mini', 'fuzzy_score': 83.33333333333334, 'new_source': 'Elvish Runemaster', 'tm_source': 'Dwarvish Runemaster', 'tm_target': 'Krasnoludzki runmistrz', 'output': 'Elfi runmistrz'}


In [27]:
client = OpenAI()
model = os.getenv("OPENAI_MODEL", "gpt-4o")
target_language = "Polish"

In [18]:
def build_translation_messages(new_source: str, target_language: str, prompt_version: str) -> list:
    """Assemble the system + user messages for the translation call."""
    
    system = PROMPTS[prompt_version].format(target_language=target_language)

    user = f"""New source (English): {new_source}"""

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

In [19]:
def call_translation(messages: list, model: str, client: OpenAI) -> str:
    """Send the messages to the API and return the translation."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip()

In [20]:
def translate_segment(new_source, target_language, prompt_version, model, client):
    """Orchestration: build messages -> call LLM -> return translation."""
    messages = build_translation_messages(new_source, target_language, prompt_version)
    output = call_translation(messages, model, client)
    return output

In [ ]:
# Example translation
prompt_version = "baseline_scratch"
new_source = gold_set_filtered[0]["query"]
print(f"String to translate: {new_source}")
translation = translate_segment(new_source, target_language, prompt_version, model, client)
print(f"Translated string: {translation}")

String to translate: Pirate Flagship
Translated string: Okręt flagowy piratów


In [ ]:
# Building cache
case_id = gold_set_filtered[0]["case_id"]
condition = "scratch"
prompt_version = "baseline_scratch"
cache = {}
key = (case_id, condition, prompt_version, model)
path = 'llm_runs.jsonl'


In [ ]:
# Not used for now
def llm_log_run(path, prompt_version, model, output):
    """Logging function to record the details of each run."""
    rec = {
        "ts": datetime.datetime.now().isoformat(timespec="seconds"),
        "prompt_version": prompt_version,
        "model": model,
        "case_id": case_id,
        "condition": condition,
        "prompt_version": prompt_version,
        "output": output,
    }
    with open(path, "a", encoding="utf-8") as f:      # append: log grows over time
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
# Translating record and saving them to llm_runs log
for record in gold_set_filtered:
    new_source = record["query"]
    output = translate_segment(new_source, target_language, "baseline_scratch", model, client)
    record["output"] = output
    record["condition"] = condition
    record["prompt_version"] = prompt_version
    record["model"] = model
    with open(path, "a", encoding="utf-8") as f:      # append: log grows over time
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


    


## Comparing the results with metrics

In [42]:
example_output = gold_set_filtered[10]["output"]
example_reference = gold_set_filtered[10]["reference"]
print(f"Example translation: {example_output} and its reference: {example_reference}")

Example translation: Jajo olbrzymiej mrówki and its reference: Jajo ogromnej mrówki


In [46]:
chrf_score = sacrebleu.sentence_chrf(example_output, [example_reference]).score
print(chrf_score)

50.36499757776167


In [49]:
# Generating CHRF score for the whole gold set
for record in gold_set_filtered:
    record["chrf"] = sacrebleu.sentence_chrf(record["output"], [record["reference"]]).score

In [50]:
# Transforming the gold set to pandas df for analysis
gold_set_df = pd.DataFrame(gold_set_filtered)
print(gold_set_df.head())

  case_id    category       edit_type  \
0     f01  fuzzy_real        reworded   
1     f02  fuzzy_real        addition   
2     f03  fuzzy_real  tag/formatting   
3     f04  fuzzy_real        addition   
4     f05  fuzzy_real        addition   

                                               query  \
0                                    Pirate Flagship   
1                                     plague ($type)   
2                            Campaigns and Scenarios   
3  <ref>dst='..calendar' text='Calendar'</ref>\n<...   
4                               Time of Day Schedule   

                                           reference target  \
0                              Okręt flagowy piratów    NaN   
1                                     zaraza ($type)    NaN   
2                             Kampanie i scenariusze    NaN   
3  <ref>dst='..calendar' text='Kalendarz'</ref>\n...    NaN   
4                                    Grafik pór dnia    NaN   

                                  tar

In [ ]:
# Grouping the results by category
gold_set_df.groupby("category")["chrf"].mean()

category
edited        49.022728
fuzzy_real    43.695667
invented      60.759897
Name: chrf, dtype: float64

In [54]:
# Grouping the results by edit_type
gold_set_df.groupby("edit_type")["chrf"].mean()

edit_type
addition          57.673206
gender            20.034964
number            47.461457
omission          54.492861
reworded          52.435540
tag/formatting    59.235678
term              42.990773
Name: chrf, dtype: float64

In [55]:
gold_set_df.groupby("edit_type")["chrf"].size()

edit_type
addition          10
gender             7
number             4
omission           3
reworded           7
tag/formatting     6
term              23
Name: chrf, dtype: int64

In [56]:
gold_set_df[gold_set_df["case_id"].isin(["f09", "e08"])][["case_id", "query", "reference", "output"]]

,case_id,query,reference,output
8,f09,female^Drake Arbiter,Smocza strażniczka,female^Drake Arbiter
37,e08,"A final blow destroys the lich, releasing a sm...","Ostateczny cios niszczy liczę, uwalniając nagł...","Ostateczny cios niszczy lichę, uwalniając małą..."


In [57]:
# Saving gold_set_df to CSV file

gold_set_df.to_csv('gold_set.csv', index=False, encoding = "utf-8")